## Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["Groq_API_KEY"]=os.getenv("GROQ_API_KEY")


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002E10A92D2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002E10A92DD30>, model_name='openai/gpt-oss-20b', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

## Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
# Create Groq model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)
# Create agent
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)

In [8]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [9]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='83ee8d3f-fb91-4dfc-8de3-f679e7fc4db1'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'The user asks: "What is 2+2?" It\'s a simple arithmetic question. The answer is 4. We should respond with the answer. Possibly also mention that it\'s 4. No extra context needed.'}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 78, 'total_tokens': 142, 'completion_time': 0.071053713, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.006891265, 'prompt_tokens_details': None, 'queue_time': 0.340919913, 'total_time': 0.077944978}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_99996fee8e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0cdfd-7e48-7bf2-8be6-1cfaf663a7f4-0', tool_calls=[], invalid_tool_calls=[], usage_me

## Token Size


In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


# Create Groq model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Create agent
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 10),
            keep=("tokens", 4)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [17]:
 #Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~326 tokens, 4 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nFind hotels in Paris.\n\n## SUMMARY\nThe user has requested assistance in locating hotels in Paris. No additional preferences or constraints have been provided.\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nSearch for suitable hotels in Paris and present options to the user.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='8bf79fc7-d690-47f2-9e70-74da11826d12'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_4226b8d0-1146-4621-93df-6090f96eee58', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 129, 'total_tokens': 168, 'completion_time': 0.042875061, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_tim

In [21]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


# Create Groq model
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

# Create agent
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.10),
            keep=("fraction", 0.04)
        )
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~213 tokens (0.1664%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='561fb93b-cf4e-4c74-a3cd-28302ea53c06'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hotels with city "Paris".', 'tool_calls': [{'id': 'fc_4b3605d2-b456-4cc1-b66b-6100e0f78ca0', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 128, 'total_tokens': 167, 'completion_time': 0.039816179, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007325032, 'prompt_tokens_details': None, 'queue_time': 0.346588155, 'total_time': 0.047141211}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_228717f27c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0ce15-165e-76b2-bed1-cd9c8f293242-0', tool_calls

* trigger → When to summarize
* keep → How much to keep
* messages → Conversation history
* tokens → Size of the conversation
* fraction → Percentage of context used
* checkpointer → Saves conversation state
* InMemorySaver → Saves it temporarily in memory
* thread_id → Identifies the conversation
* middleware → Manages the agent's execution

Your code creates a **LangChain AI agent using Groq's GPT-OSS-20B model**, gives it a `search_hotels` tool to return hotel information for different cities, and uses `InMemorySaver` to maintain the conversation state through a `thread_id`. The `SummarizationMiddleware` automatically manages the growing conversation history by summarizing older messages when the context reaches the configured fraction and keeping recent messages for continued conversation. The loop sends hotel queries for six cities using the same conversation thread, while `count_tokens()` roughly estimates the number of tokens in the returned messages so you can observe how the conversation size changes and understand when summarization is being applied.


## Human in the loop Middleware 

Pause agent execution for human approval,editing or rejection tools call before they execute.Human-in-the-loop is useful for the following:
* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.


In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [5]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [6]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fe71debf-1c03-46a5-8ec5-a16d1f446d9a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_658da48d-5c3b-4fc7-b8c8-4378176fb46d', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.049361946, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009833648, 'prompt_tokens_details': None, 'queue_time': 0.463705788, 'total_time': 0.059195594}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_24bfb4a850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id=

In [7]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to john@test.com with subject “Hello”.


In [8]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='fe71debf-1c03-46a5-8ec5-a16d1f446d9a'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_658da48d-5c3b-4fc7-b8c8-4378176fb46d', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.049361946, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009833648, 'prompt_tokens_details': None, 'queue_time': 0.463705788, 'total_time': 0.059195594}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_24bfb4a850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id=

## Reject


In [11]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry, but I can’t send the email automatically. If you’d like, I can draft the message for you or let me know if you’d like me to try again.
